In [ ]:
#  model load

In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
# 1. Load variables from the .env file
load_dotenv()
OLLAMA_API_KEY = os.environ.get("OLLAMA_API_KEY")
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY")



llm1 = init_chat_model("google_genai:gemini-2.5-flash")
response = llm1.invoke("What is the color of the sky answer in one word?")
print(response.content)


llm2 = init_chat_model("ollama:nemotron-3-super:cloud",base_url="https://ollama.com")
response = llm2.invoke("What is the color of the sky answer in one word?")
print(response.content)


Blue
Blue


# middle ware

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware



agent = create_agent(
    model=llm1,
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model=llm2,
        trigger=("tokens", 30),
        keep=("messages",1)
    )],
)

In [5]:
from langchain.messages import HumanMessage, AIMessage

from pprint import pprint

response = agent.invoke(
    {
        "messages":[
            HumanMessage(content="who is mahatama Gandhi?"),
            AIMessage(content="mahatama gandhi was the father of nation of india."),
            HumanMessage(content="when did he born?"),
            AIMessage(content="He born on 2 October 1869 in Porbandar, India."),
            HumanMessage(content="what is his nationality?"),
            AIMessage(content="He was an Indian independence activist."),
            HumanMessage(content="what is his famous quote?"),
            AIMessage(content="His famous quote is 'Be the change that you wish to see in the world.'"),
            HumanMessage(content="what is his death date?"),
        ]
    },
    {"configurable": {"thread_id": "1"}}
)

pprint(response)


{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user’s goal is to obtain basic biographical information about Mahatma Gandhi, including his identity, date of birth, nationality, and a notable quote.\n\n## SUMMARY\n- Mahatma Gandhi is described as the "father of the nation of India."\n- He was born on 2\u202fOctober\u202f1869 in Porbandar, India.\n- His nationality is identified as Indian, and he is noted as an Indian independence activist.\n- His famous quote is: “Be the change that you wish to see in the world.”\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone—no further actions are required based on the conversation.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='54c55982-462f-4404-b312-7c570910012f'),
              HumanMessage(content='what is his death date?', additional_kwargs={}, response_metadata={}, id='0715987f-56dd-4536-af50-7b8d924110d1'),
              AIMessage(content='Mahatma Gandhi 

In [7]:
for message in response["messages"]:
    print(f"{message.type}: {message.content}") 
    print("-------------------------------------------")


human: Here is a summary of the conversation to date:

## SESSION INTENT
The user’s goal is to obtain basic biographical information about Mahatma Gandhi, including his identity, date of birth, nationality, and a notable quote.

## SUMMARY
- Mahatma Gandhi is described as the "father of the nation of India."
- He was born on 2 October 1869 in Porbandar, India.
- His nationality is identified as Indian, and he is noted as an Indian independence activist.
- His famous quote is: “Be the change that you wish to see in the world.”

## ARTIFACTS
None

## NEXT STEPS
None—no further actions are required based on the conversation.
-------------------------------------------
human: what is his death date?
-------------------------------------------
ai: Mahatma Gandhi died on **30 January 1948**.
-------------------------------------------


## human in the loop

In [8]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body"""
    # fake email sending
    return f"Email Sent"

In [9]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model=llm1,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)


In [10]:

from langchain.messages import HumanMessage

config = {"configurable":{"thread_id":"1"}}

response = agent.invoke(
    {
        "messages":[HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Sean, I am going to be late for our meeting tomorrow. Can we reschedule? Best, John." # We are Sean and we have got this message to reschedule from John
    },
    config=config
)

In [11]:
response

{'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='7c127999-5b6b-4615-adc9-dc916fd25b67'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'a999aeff-d662-4580-b486-9053afad4ddc': 'CugCAQw51sfgCs8D1LIQvaorUl8+R62ouq80w8iyS7PUGueeqE2qBCZ7MmlbWXsWACDmoAsJ3FMaLlw/PGJ8YBNCVtLmpboHS+inDH9SSHsda/25krW7tvVNiEfnj0phP+siJ6otF3ewh6jP1ox5sG0dWiDntqDCN3/tMH4JYJ4kWw+oiZfJ2piyRw8Z+mf2jCjaN/H+NjCnVA1mbb5rGMqNKC7CjXTYRIsWnMxAufazAVubvlVbSOXUxVM19QcbZd1EHMpLFsml8aNhOy1hjlNlunUoEObzXxOSXLAkXPGobbgIhS9Hx+y5oYVgG6hkKAv3W7Or6bkzugnJ4QQYByJxeGi/bLC4yMnnm6aHPbkHyrEt1T+Ku4h5DgGzGzlKkjgka3bE6nblM4M0yYnSb1LnVOlT11CinBxKBRsv6WaJbX+Ag1amU+qubKHtwW4P0HfMmQWpQ1EYHGamZYOjLwQM4Uh9mFMSjQ7j'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 

In [12]:
from pprint import pprint
pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'Thanks '
                                                                          'for '
                                                                          'letting '
                                                                          'me '
                                                                          'know. '
                                                                          'Yes, '
                                                                          'we '
                                                                          'can '
                                                                          'definitely '
                                                                          'reschedule. '
    

In [13]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'Hi John, Thanks for letting me know. Yes, we can definitely reschedule. What time works best for you? Best, Sean.'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'Hi John, Thanks for letting me know. Yes, we can definitely reschedule. What time works best for you? Best, Sean.'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='11451219b8c93775604c4cc380ae701f')]


In [14]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, Thanks for letting me know. Yes, we can definitely reschedule. What time works best for you? Best, Sean.


In [15]:
print(response['__interrupt__'][0].value['review_configs'])

[{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]


## now approve this message

In [16]:
from langgraph.types import Command

response = agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config
)

pprint(response)

{'email': 'Hi Sean, I am going to be late for our meeting tomorrow. Can we '
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='7c127999-5b6b-4615-adc9-dc916fd25b67'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'a999aeff-d662-4580-b486-9053afad4ddc': 'CugCAQw51sfgCs8D1LIQvaorUl8+R62ouq80w8iyS7PUGueeqE2qBCZ7MmlbWXsWACDmoAsJ3FMaLlw/PGJ8YBNCVtLmpboHS+inDH9SSHsda/25krW7tvVNiEfnj0phP+siJ6otF3ewh6jP1ox5sG0dWiDntqDCN3/tMH4JYJ4kWw+oiZfJ2piyRw8Z+mf2jCjaN/H+NjCnVA1mbb5rGMqNKC7CjXTYRIsWnMxAufazAVubvlVbSOXUxVM19QcbZd1EHMpLFsml8aNhOy1hjlNlunUoEObzXxOSXLAkXPGobbgIhS9Hx+y5oYVgG6hkKAv3W7Or6bkzugnJ4QQYByJxeGi/bLC4yMnnm6aHPbkHyrEt1T+Ku4h5DgGzGzlKkjgka3bE6nblM4M0yYnSb1LnVOlT11CinBxKBRsv6WaJbX+Ag1amU+qubKHtwW4P0HfMmQWpQ1EYHGa

In [ ]:
from pprint import pprint
pprint(response)

{'email': 'Hi Sean, I am going to be late for our meeting tomorrow. Can we '
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='7c127999-5b6b-4615-adc9-dc916fd25b67'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'a999aeff-d662-4580-b486-9053afad4ddc': 'CugCAQw51sfgCs8D1LIQvaorUl8+R62ouq80w8iyS7PUGueeqE2qBCZ7MmlbWXsWACDmoAsJ3FMaLlw/PGJ8YBNCVtLmpboHS+inDH9SSHsda/25krW7tvVNiEfnj0phP+siJ6otF3ewh6jP1ox5sG0dWiDntqDCN3/tMH4JYJ4kWw+oiZfJ2piyRw8Z+mf2jCjaN/H+NjCnVA1mbb5rGMqNKC7CjXTYRIsWnMxAufazAVubvlVbSOXUxVM19QcbZd1EHMpLFsml8aNhOy1hjlNlunUoEObzXxOSXLAkXPGobbgIhS9Hx+y5oYVgG6hkKAv3W7Or6bkzugnJ4QQYByJxeGi/bLC4yMnnm6aHPbkHyrEt1T+Ku4h5DgGzGzlKkjgka3bE6nblM4M0yYnSb1LnVOlT11CinBxKBRsv6WaJbX+Ag1amU+qubKHtwW4P0HfMmQWpQ1EYHGa

In [18]:
pprint(response["messages"][-1].content)

[{'extras': {'signature': 'Cs0BAQw51sdmda41wvQfA+jUn3deyjOWw8QrkAJAmso9g6Bs+3J/VKOxZt44GCBNY93Kq8SzgBeDnCeG6tRpH4Iy7j/SjJYLV0HnpTRdiSI1L0AGk93fR31xeB28hfjc4yiekTbHVHgkZ8yaiSD77HWGm1GktVa1nCm5mBMUFzoz4X/59Mqr71dVAXJZPxCrW9Tj0YgWqFvaP4R7vwT2dt8iycvutEvVkdMAgM9WP3WBCVEMBKRNHWcFEkNkZqZoV5fOmU6873f0vJaKOKXZcA=='},
  'text': 'I have read your email and sent a response in the same thread.',
  'type': 'text'}]


In [19]:
## rejecting interupt

In [20]:
config = {"configurable":{"thread_id":"2"}}

response = agent.invoke(
    {
        "messages":[HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Sean, I am going to be late for our meeting tomorrow. Can we reschedule? Best, John." # We are Sean and we have got this message to reschedule from John
    },
    config=config
)



In [21]:
response

{'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='152652fc-4392-43e4-adbc-79c38e9836ba'),
  AIMessage(content=[{'type': 'text', 'text': 'I have read your email. What would you like the reply to say?', 'extras': {'signature': 'CtQFAQw51sfulWaQ0On4OeogVjKGuk+Ga7aEifmyOlRfCVFxxpyBgzSWoZr1xeNvyzBpn+A5xMdht+elZ4o42CDXnYwe1oGXXKyk5wDpoEqUhGwM0BOKS1JihE856xp/rmQPiEbaBeceRFkUC7PpZI9hrorZRv7z4IcqIg7dsyLOenjeSRoVU+7GJBCDMl6+qadmxhLHyaUl+arvWqgEdUUb67thsoNKn1QCKibgiu0vbtGWFQp89R5neEamMujQvc3cvpVuIfqh/vM6NqWiuzJaw1pl7qWxjhw3mE7DUzYgzekQgrsFvoF0CV++/KjnfVEmLJ8kPEqpxSV581xY1ssua7Q9dqfu3+c2R4LZgfI7XaZM+r/vARnGpjIoPQzGWOztSF8alLfhYmn8wT2cWz9ELaHTSmqlGvqJRDFlXjYA0T/DMXIsV6ZVzFYUZzPtqfVWP57EEMBibGURhxUqyp3u64FmHgmtRBQD4EGufZ28H3642mX+oe48e/j+f+ZA9irg7zBzO7Ghv5OGV1Rjhp6K4B8OIlm4RsblnNwJVH6I02QY/Dz/vEGfQW9rF0d7ZuapjU9ucRoxLPavIh6shXzuHvPDGop7IMGPmlKOReYzIvq1p5BrqVIC59CYnOkwcYPA+

In [22]:
pprint(response["messages"][-1].content)

[{'extras': {'signature': 'CtQFAQw51sfulWaQ0On4OeogVjKGuk+Ga7aEifmyOlRfCVFxxpyBgzSWoZr1xeNvyzBpn+A5xMdht+elZ4o42CDXnYwe1oGXXKyk5wDpoEqUhGwM0BOKS1JihE856xp/rmQPiEbaBeceRFkUC7PpZI9hrorZRv7z4IcqIg7dsyLOenjeSRoVU+7GJBCDMl6+qadmxhLHyaUl+arvWqgEdUUb67thsoNKn1QCKibgiu0vbtGWFQp89R5neEamMujQvc3cvpVuIfqh/vM6NqWiuzJaw1pl7qWxjhw3mE7DUzYgzekQgrsFvoF0CV++/KjnfVEmLJ8kPEqpxSV581xY1ssua7Q9dqfu3+c2R4LZgfI7XaZM+r/vARnGpjIoPQzGWOztSF8alLfhYmn8wT2cWz9ELaHTSmqlGvqJRDFlXjYA0T/DMXIsV6ZVzFYUZzPtqfVWP57EEMBibGURhxUqyp3u64FmHgmtRBQD4EGufZ28H3642mX+oe48e/j+f+ZA9irg7zBzO7Ghv5OGV1Rjhp6K4B8OIlm4RsblnNwJVH6I02QY/Dz/vEGfQW9rF0d7ZuapjU9ucRoxLPavIh6shXzuHvPDGop7IMGPmlKOReYzIvq1p5BrqVIC59CYnOkwcYPA+TLqvvYjbUYvUhLrpabkE/buDzC2xWU7kSQtJSkppNvf+m+IQLXz/PjQ6Hkr+bK6odmxQBws8Lh1kEQxkQTh1nMAnyowlAmHc+Rqh6wfQQ16VICsJluRu3Ve3JbaSSl2gwd89GvL7PQo0acrUw0BZC257+IGBz50Ft6m1IDFFa1fQ4hSg+1TdsSVMOm+nCSIfh+q6weBPpaHAxGVfVWRIjaoKnxg9bARDzgA461qAsiADY/BgYobDgQl2x2vfTjRyv5KQ7tlaJb1Jbjc9iIhgPE4WFLo74KaRYZIB0zh3jvKxTQ21sPjOBLy8C6FOser4xRmaQ=='